In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

SEEDS = [42, 7, 123, 13, 99, 2024]
N_SPLITS = 10
print(f'Seeds: {SEEDS}')

Seeds: [42, 7, 123, 13, 99, 2024]


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA  = pd.read_csv('test-data.csv', index_col='id')
print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')

Train: (13249, 41), Test: (8834, 41)


In [4]:
# ── Cell 4: Preprocessing ────────────────────────────────────────────────────
def preprocess(df):
    df = df.copy()

    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent',
        'birth_defects',   # MI=0.000, pure noise
        'alive',           # MI=0.000, pure noise
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender','risk_level',
                'heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']       = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']        = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']        = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']       = df['place_birth'].map({'I': 1, 'H': 0})
    df['gender']            = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']           = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # ── Core engineered features ─────────────────────────────────────────────
    df['defect_sum']           = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum']          = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']            = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']             = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']        = (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                                  - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym']         = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                                  df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                                  df['symptom_5'].fillna(0)*2)
    df['both_parents_defect']  = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']     = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    # ── Class 4 amplifiers (63.8% zero-symptom rate) ─────────────────────────
    df['zero_symptom']  = (df['symptom_sum'] == 0).astype(int)
    df['zero_defect']   = (df['defect_sum']  == 0).astype(int)
    df['zero_both']     = ((df['symptom_sum'] == 0) & (df['defect_sum'] == 0)).astype(int)
    df['very_low_sym']  = (df['symptom_sum'] <= 1).astype(int)

    # ── NEW: Zone fingerprints from EDA ──────────────────────────────────────
    # Each zone maps to a natural cluster of classes found in the data:
    # cls 0,2,8 → high sym + high def | cls 5,6 → low sym + low def
    # cls 3,7   → mid sym + mid def   | cls 9,1 → mid sym + higher def
    df['high_sym_high_def'] = ((df['symptom_sum'] >= 4) & (df['defect_sum'] >= 2)).astype(int)   # cls 0,2,8
    df['mid_sym_low_def']   = (df['symptom_sum'].between(1,2) & (df['defect_sum'] <= 1)).astype(int)  # cls 5,6
    df['mid_sym_mid_def']   = (df['symptom_sum'].between(2,3) & df['defect_sum'].between(1,2)).astype(int)  # cls 3,7
    df['mid_sym_high_def']  = (df['symptom_sum'].between(2,4) & (df['defect_sum'] >= 2)).astype(int)  # cls 9,1
    df['full_load']         = ((df['symptom_sum'] >= 4) & (df['defect_sum'] >= 3)).astype(int)    # cls 8 fingerprint

    # ── NEW: symptom_5 is the strongest 7-vs-9 separator (diff=0.137) ────────
    df['sym5_x_def']    = df['symptom_5'].fillna(0) * df['defect_sum']

    # ── NEW: sym/def grid interaction ─────────────────────────────────────────
    df['sym_def_zone']  = (df['symptom_sum'] * 5 + df['defect_sum']).astype(int)

    # ── NEW: missing pattern signals ─────────────────────────────────────────
    df['all_sym_missing'] = (df[['symptom_1','symptom_2','symptom_3',
                                  'symptom_4','symptom_5']].isna().sum(axis=1) == 5).astype(int)
    df['maternal_def_missing'] = df['maternal_defect'].isna().astype(int)

    df = df.fillna(-1)
    return df

X_train_raw = preprocess(TRAIN_DATA)
X_test_raw  = preprocess(TEST_DATA)
y_train     = TRAIN_LABEL['disorder'].values

DROP_ADV = ['blood_cell_count', 'white_blood_cell_count', 'mother_age']
X_train = X_train_raw.drop(columns=DROP_ADV)
X_test  = X_test_raw.drop(columns=DROP_ADV)
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'New features added: high_sym_high_def, mid_sym_low_def, mid_sym_mid_def,')
print(f'                    mid_sym_high_def, full_load, sym5_x_def,')
print(f'                    sym_def_zone, all_sym_missing, maternal_def_missing')


X_train: (13249, 67), X_test: (8834, 67)
New features added: high_sym_high_def, mid_sym_low_def, mid_sym_mid_def,
                    mid_sym_high_def, full_load, sym5_x_def,
                    sym_def_zone, all_sym_missing, maternal_def_missing


In [5]:
# ── Cell 5: Class weights + params ───────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

PARAMS = {
    'iterations'           : 3000,
    'learning_rate'        : 0.03,
    'depth'                : 8,
    'l2_leaf_reg'          : 1.2554074561515052,
    'random_strength'      : 0.15300699009014024,
    'rsm'                  : 0.6693037260566354,
    'bagging_temperature'  : 0.7561917100518147,
    'min_data_in_leaf'     : 23,
    'early_stopping_rounds': 150,
    'eval_metric': 'TotalF1:average=Macro',   # ← was 'BalancedAccuracy' (binary only!)
    'verbose'              : 0,
    'thread_count'         : -1,
}
print('Params ready.')

Params ready.


In [6]:
# ── Cell 6: Train — 6 seeds x 10 folds ──────────────────────────────────────
print(f'Training {len(SEEDS)} seeds x {N_SPLITS} folds')

all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test), 10))
seed_oof_scores = []

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba  = np.zeros((len(y_train), 10))
    test_preds = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = CatBoostClassifier(
            class_weights=class_weights,
            random_seed=SEED,
            **PARAMS
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    seed_oof_scores.append(oof_score)
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    all_oof_proba  += oof_proba / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'Per-seed OOF: {[round(s,4) for s in seed_oof_scores]}')
print(f'Seed std:     {np.std(seed_oof_scores):.4f}')
print(f'FINAL OOF BA: {final_oof:.4f}')
print(f"{'='*60}")

Training 6 seeds x 10 folds

======================================== SEED=42 ========================================
  Fold  1: BA=0.3804  best_iter=52
  Fold  2: BA=0.4147  best_iter=49
  Fold  3: BA=0.4148  best_iter=48
  Fold  4: BA=0.3772  best_iter=184
  Fold  5: BA=0.4034  best_iter=127
  Fold  6: BA=0.4122  best_iter=107
  Fold  7: BA=0.3598  best_iter=197
  Fold  8: BA=0.3903  best_iter=49
  Fold  9: BA=0.4066  best_iter=44
  Fold 10: BA=0.4364  best_iter=10
  OOF BA (seed=42): 0.3992 | mean=0.3996 ± 0.0214

======================================== SEED=7 ========================================
  Fold  1: BA=0.4049  best_iter=74
  Fold  2: BA=0.4052  best_iter=44
  Fold  3: BA=0.4275  best_iter=108
  Fold  4: BA=0.4194  best_iter=77
  Fold  5: BA=0.3745  best_iter=80
  Fold  6: BA=0.4274  best_iter=26
  Fold  7: BA=0.3684  best_iter=113
  Fold  8: BA=0.3670  best_iter=69
  Fold  9: BA=0.3953  best_iter=149
  Fold 10: BA=0.3775  best_iter=23
  OOF BA (seed=7): 0.3967 | mean=0

In [7]:
# ── Cell 7: Per-class recall ─────────────────────────────────────────────────
best_recall = {0:0.411, 1:0.362, 2:0.296, 3:0.319, 4:0.759,
               5:0.309, 6:0.503, 7:0.253, 8:0.560, 9:0.175}

oof_labels = np.argmax(all_oof_proba, axis=1)
report = classification_report(y_train, oof_labels, output_dict=True)
print(f'OOF BA: {final_oof:.4f}\n')
print(f'{"Class":<6} {"best LB run":>12} {"this run":>10} {"Δ":>7}')
print('-' * 40)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = best_recall[cls]
    delta = r - r_old
    flag  = ' ← UP' if delta > 0.02 else (' ← LOW' if r < 0.25 else '')
    print(f'{cls:<6} {r_old:>12.3f} {r:>10.3f} {delta:>+7.3f}{flag}')

OOF BA: 0.3863

Class   best LB run   this run       Δ
----------------------------------------
0             0.411      0.411  +0.000
1             0.362      0.372  +0.010
2             0.296      0.319  +0.023 ← UP
3             0.319      0.318  -0.001
4             0.759      0.638  -0.121
5             0.309      0.345  +0.036 ← UP
6             0.503      0.533  +0.030 ← UP
7             0.253      0.248  -0.005 ← LOW
8             0.560      0.516  -0.044
9             0.175      0.162  -0.013 ← LOW


In [8]:
# ── Cell 8: Save submission ───────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({'id': TEST_DATA.index, 'disorder': final_preds}).set_index('id')
submission.to_csv('submission_eval2.csv')
print('Saved: submission_eval2.csv')
print(f'\nFinal OOF: {final_oof:.4f}')

Saved: submission_eval2.csv

Final OOF: 0.3863
